# Fit Model parameters to single variable data (voltage)

This notebook follows the [PBPram example of the same name](https://github.com/paramm-team/pybamm-param/blob/develop/examples/notebooks/datafit_single_variable.ipynb)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import pybop
import pybamm
pybop.plot.PlotlyManager().pio.renderers.default = "notebook_connected"

# 1. Load the data
The first step is to load the data to which we want to fit the model. For this example, we use the experimental dataset which from [Brosa Planella et al. (2021) article](https://www.sciencedirect.com/science/article/pii/S0013468621008148). This data set is already in the right format, if you are using your own dataset you should ensure that the headers of the relevant columns match the variable names in PyBaMM (e.g. `"Time [s]"`, `"Voltage [V]"`...)

In [ ]:
# Data is up two directory then in data/pybamm/
data_path = Path.cwd().parent / "data" / "pybamm"/"LGM50_789_1C_25degC.csv"
data = pd.read_csv(data_path)

# This data has some issues in the time column, so we will fix it
mask = data["Time [s]"].values[:-1] < data["Time [s]"].values[1:]  # Check if the time is increasing

# where the mask is false, we will drop the row
data = data.iloc[:-1][mask]


# Transform the data into a pybop dataset
dataset = pybop.Dataset(
    {
     "Time [s]": data["Time [s]"].values,
     "Voltage [V]": data["Voltage [V]"].values,
     "X-averaged cell temperature [K]": data["X-averaged cell temperature [K]"].values,
    }
)


In [ ]:
# Note this is to correct for the lack of current data in the dataset
# Find the point at which the voltage is at its minimum and get the index
min_voltage = dataset["Voltage [V]"].min()
min_voltage_index = dataset["Voltage [V]"].argmin()
# Generate a current profile that will discharge the battery to the minimum voltage at 5A then 0A to the end
current_profile = np.zeros_like(dataset["Time [s]"])
current_profile[:min_voltage_index] = 5
dataset.__setitem__("Current function [A]", current_profile)


# 2. Define the model
Next we need to define the model we want to fit to the data. This also includes defining the solver, spatial methods, parameters that we are not fitting, and operating conditions. To streamline this, we wrap everything into a PyBaMM simulation. The basic idea is that the simulation already includes everything needed to solve the model under certain conditions.

In this case we choose the Single Particle Model (SPM) with a contact resistance, which we will fit to the data.

In [ ]:
# Define the model and parameter set
parameter_set = pybop.ParameterSet.pybamm("Chen2020")

model = pybop.lithium_ion.SPM(parameter_set=parameter_set, options={"contact resistance": "true"})


# 3. Define the Optimisation Problem 

In [ ]:
# Define the optimization parameters with initial values and bounds
parameters = pybop.Parameters(
    pybop.Parameter(
        "Negative particle diffusivity [m2.s-1]",
        initial_value=5e-14,
        bounds=(2.06e-16, 2.06e-12),
    ),
    pybop.Parameter(
        "Contact resistance [Ohm]",
        initial_value=0,
        bounds=(0, 0.5),
    ),
)
parameters.update() 

In [ ]:
problem = pybop.FittingProblem(model=model, parameters=parameters, dataset=dataset)
cost = pybop.SumSquaredError(problem)

In [ ]:
optim = pybop.SciPyMinimize(
    cost,
    method="Nelder-Mead",
)

In [ ]:
results = optim.run()

In [ ]:
# Plot the results from pybop
pybop_fig = pybop.plot.quick(problem=problem)

In [ ]:
# Construct a plot using the pbparam minimization results
pbparam_fig = pybop.plot.quick(problem=problem, problem_inputs=[1.849e-14, 0.020944])

# Experiment Fit
 The Above fits on a current function only below we modify the codebase considerably and fit on an experiment (similar to pb-param)

In [ ]:
# Define the operating conditions
experiment = pybamm.Experiment(
    [
        "Discharge at 1C until 2.5 V",
        "Rest for 2 hours",
    ],
    period="30 seconds",
)
solution = model.predict(experiment=experiment)
solution.plot(["Current [A]", "Voltage [V]"])

In [ ]:
for i in parameters.keys():
    print(i)
    print(f"{i}: {model.parameter_set[i]}")

In [ ]:
test_parameters = parameters.as_dict()
test_parameters["Negative particle diffusivity [m2.s-1]"] = 1.849e-14
test_parameters["Contact resistance [Ohm]"] = 0.021

In [ ]:
test_parameters_list = parameters.rvs(10)

In [ ]:
test_parameters_list_of_dicts = []
for i in test_parameters_list:
    test_parameters_list_of_dicts.append(
        {
            "Negative particle diffusivity [m2.s-1]": i[0],
            "Contact resistance [Ohm]": i[1],
        }
    )


In [ ]:
solution = model.simulate(t_eval=np.arange(0,10645), inputs=[1.849e-14, 0.021])

In [ ]:
solution.data

In [ ]:
solution.plot(["Current [A]", "Voltage [V]"])

In [ ]:
model.build()
solution = model.simulate(t_eval=np.arange(0,10645), inputs=test_parameters_list_of_dicts)

In [ ]:
solution.plot(["Current [A]", "Voltage [V]"])

In [ ]:
optim.cost([5e-14, 0])

In [1]:
class A:
    def __init__(self):
        self.X = "A"
        print("A")

    def a_print(self):
        print("cls A")

In [2]:
class _A:
    def __init__(self):
        self.X = "_A"
        print("_A")

    def a_print(self):
        print("cls _A")


In [ ]:
class B(A=A):
    def __new__(cls, other=None):
        if other is not None:
            print("Other")
            return other().__new__(cls)
        else:
            print("Normal")
            return super().__new__(cls)

    def __init__(self, other=None):
        super().__init__()

        self.Y = "B" 
        print("B")
    
    def b_print(self):
        print("cls B", self.Y)

TypeError: B.__init_subclass__() takes no keyword arguments

In [7]:
my_b = B(other=_A)

Other
_A
A
B


In [8]:
my_b.a_print()

cls A


In [9]:
my_b.b_print()

cls B B


In [ ]:
#my_b.a_print()
my_b.b_print()